# 00 — Environment setup (Arch Linux + CUDA)

Run this notebook first, top to bottom, before any other notebook in `arch-cuda/`.
It exists to catch GPU/driver/library mismatches early instead of discovering
them halfway through a training run.

Detected on this machine when this pipeline was written:

| Component | Value |
|---|---|
| OS | Arch Linux (rolling) |
| GPU | NVIDIA GeForce RTX 5060 Laptop GPU, 8 GB VRAM |
| Compute capability | `sm_120` (Blackwell) |
| Driver | 610.43.03 |
| System CUDA toolkit | 13.3 |
| System cuDNN | 9.24 |
| Python | 3.14.6 |
| Package manager | `uv` (no conda/mamba on this machine) |

The original scripts in `dev/programming/2026/Tufts-ai-final/` target macOS
(`device='mps'`, `yolov8n.pt`). Every notebook here uses `device='cuda'` and
`yolov8s.pt` instead.


## 1. One-time terminal setup

Run this **outside** Jupyter, once, from the repo root:

```bash
cd arch-cuda
uv venv .venv                 # uses the system Python (3.14.6) by default
source .venv/bin/activate
uv pip install -r <(python - <<'PY'
import tomllib
with open("pyproject.toml", "rb") as f:
    deps = tomllib.load(f)["project"]["dependencies"]
print("\n".join(deps))
PY
)
uv pip install ipykernel
python -m ipykernel install --user --name smartsusan-cuda --display-name "smartsusan (CUDA)"
```

Then select the **"smartsusan (CUDA)"** kernel for this and every other
notebook in `arch-cuda/`.

If you'd rather install straight from the notebook, run the guarded cell
below (`RUN_INSTALL = True`) instead of the terminal block — useful the first
time you open this in a fresh kernel.


In [ ]:
RUN_INSTALL = False  # flip to True once, then back to False

if RUN_INSTALL:
    %pip install -q ultralytics torch torchvision rembg onnxruntime-gpu \
        pillow pyyaml matplotlib numpy opencv-python-headless \
        selenium undetected-chromedriver requests


## 2. Confirm the GPU is visible to the OS/driver stack

(Independent of any Python package — this only depends on the NVIDIA driver.)


In [ ]:
import subprocess

def run(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout or p.stderr)

run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total,memory.used,compute_cap", "--format=csv"])
run(["nvcc", "--version"])


## 3. Intermediary test — PyTorch actually sees the GPU

This is the gate for every later notebook. `sm_120` (Blackwell, RTX 50-series)
is new enough that some PyTorch builds only ship generic/PTX kernels for it
rather than a compiled `sm_120` kernel, which silently falls back to slow JIT
compilation or CPU. The cell below checks both "is CUDA available" and "can it
actually run a kernel," and tells you what to do if either fails.


In [ ]:
import torch

print("torch version:", torch.__version__)
print("torch was built with CUDA:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))
    print("device capability:", torch.cuda.get_device_capability(0))
else:
    raise RuntimeError(
        "torch.cuda.is_available() is False. Fix before continuing:\n"
        "  uv pip uninstall torch torchvision\n"
        "  uv pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128\n"
        "(or the newest cuXXX index at https://pytorch.org/get-started/locally/ "
        "if cu128 predates sm_120 support by the time you read this)."
    )


In [ ]:
import time

# Real kernel execution, not just availability — this is what would silently
# fall back to CPU (or hang recompiling PTX) if sm_120 isn't properly supported.
size = 4096
a = torch.randn(size, size, device="cuda")
b = torch.randn(size, size, device="cuda")

torch.cuda.synchronize()
t0 = time.time()
for _ in range(10):
    c = a @ b
torch.cuda.synchronize()
gpu_s = time.time() - t0

a_cpu, b_cpu = a.cpu(), b.cpu()
t0 = time.time()
for _ in range(10):
    c_cpu = a_cpu @ b_cpu
cpu_s = time.time() - t0

print(f"GPU matmul: {gpu_s:.3f}s  |  CPU matmul: {cpu_s:.3f}s  |  speedup: {cpu_s / gpu_s:.1f}x")
assert gpu_s < cpu_s, "GPU was not faster than CPU — CUDA kernels are probably not actually running on-device."
print("OK: GPU is executing kernels and is faster than CPU.")


## 4. Intermediary test — Ultralytics + ONNX Runtime GPU provider

`ultralytics` re-checks its own environment; `onnxruntime-gpu` needs its own
sanity check because it silently falls back to CPU if the CUDA execution
provider can't load (used later for GPU-accelerated background removal).


In [ ]:
import ultralytics
ultralytics.checks()


In [ ]:
import onnxruntime as ort

providers = ort.get_available_providers()
print("available onnxruntime providers:", providers)
assert "CUDAExecutionProvider" in providers, (
    "CUDAExecutionProvider not available — rembg background removal will run "
    "on CPU only. Check `onnxruntime-gpu` (not plain `onnxruntime`) is installed "
    "and that it matches the system CUDA/cuDNN versions."
)
print("OK: CUDAExecutionProvider is available.")
